# MetadataManager Tutorial: Enriching Data with Condition Metadata

This tutorial demonstrates how to create enriched metadata views by combining:
1. Actual data fields from HuggingFace datasets (via HfQueryAPI)
2. Condition metadata from DataCard definitions

We'll use the **BrentLab/harbison_2004** ChIP-chip dataset, which has 14 experimental conditions with detailed metadata about media composition, temperature, and other environmental factors.

## Why Enrich Metadata?

The raw data contains a `condition` field with values like "YPD", "HEAT", "GAL". But to analyze data by media type or carbon source, we need to:
- Extract nested information from condition definitions
- Create queryable columns for filtering and grouping
- Join this enriched metadata with the actual measurements

This tutorial teaches you how to:
1. Use DataCard to explore nested condition metadata
2. Manually construct SQL CASE expressions to enrich your data
3. Create metadata views that combine data with DataCard information

## 1. Setup and Imports

In [32]:
from tfbpapi.datainfo import DataCard
from tfbpapi import HfQueryAPI
import duckdb
import pandas as pd
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 2. Explore Condition Metadata with DataCard

First, let's load the DataCard and explore what condition metadata is available.

In [33]:
# Load DataCard
card = DataCard('BrentLab/harbison_2004')

# Get the config to understand the data structure
config = card.get_config('harbison_2004')
print(f"Dataset type: {config.dataset_type}")
print(f"Number of features: {len(config.dataset_info.features)}")
print(f"\nFeature names:")
for feature in config.dataset_info.features:
    print(f"  - {feature.name} ({feature.dtype})")

Dataset type: DatasetType.ANNOTATED_FEATURES
Number of features: 7

Feature names:
  - condition ({'class_label': {'names': ['YPD', 'SM', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90', 'Thi-', 'GAL', 'HEAT', 'Pi-', 'RAFF']}})
  - regulator_locus_tag (string)
  - regulator_symbol (string)
  - target_locus_tag (string)
  - target_symbol (string)
  - effect (float64)
  - pvalue (float64)


### Extract Media Specifications Using get_field_attribute()

The new `get_field_attribute()` method makes it easy to extract nested specifications from field definitions. Let's use it to explore the media specifications for each condition.

In [34]:
# Extract media specifications for all conditions
media_specs = card.get_field_attribute('harbison_2004', 'condition', 'media')

print(f"Found media specifications for {len(media_specs)} conditions\n")

# Display them in a readable format
for condition in sorted(media_specs.keys()):
    print(f"\n{condition}:")
    print(json.dumps(media_specs[condition], indent=2))

AttributeError: 'DataCard' object has no attribute 'get_field_attribute'

### Understanding the Nested Structure

Notice that each media specification contains:
- `name`: The media name (e.g., "YPD", "synthetic_complete")
- `carbon_source`: Either a list of dicts with compound/concentration, or "unspecified"
- `nitrogen_source`: Either a list of dicts with compound/concentration, or "unspecified"
- Sometimes `additives`: Additional compounds like butanol

To create queryable metadata columns, we need to flatten these nested structures.

## 3. Load Data with HfQueryAPI

Now let's load the actual data using HfQueryAPI. We'll create a shared DuckDB connection that we can use for building our enriched metadata view.

In [ ]:
# Create shared DuckDB connection
conn = duckdb.connect(':memory:')

# Initialize HfQueryAPI with shared connection
api = HfQueryAPI('BrentLab/harbison_2004', duckdb_conn=conn)

# Load the harbison_2004 config data
api.load_config('harbison_2004')

print("Data loaded successfully!")
print("\nAvailable views in DuckDB:")
views = conn.execute("SHOW TABLES").fetchall()
for view in views:
    print(f"  - {view[0]}")

### Examine the Base Data

Let's look at the structure of the base data to see what fields are available.

In [ ]:
# Query the base view
base_df = conn.execute("""
    SELECT *
    FROM harbison_2004_train
    LIMIT 5
""").df()

print(f"Base data shape: {base_df.shape}")
print(f"\nColumn names: {list(base_df.columns)}")
print("\nSample rows:")
display(base_df)

In [ ]:
# Check unique conditions in the data
conditions_df = conn.execute("""
    SELECT DISTINCT condition
    FROM harbison_2004_train
    ORDER BY condition
""").df()

print(f"Unique conditions in data: {len(conditions_df)}")
print(conditions_df['condition'].tolist())

## 4. Flatten Media Specifications

Before we can build SQL CASE expressions, we need to flatten the nested media specifications into simple strings. We'll create helper functions to extract:
- Media name
- Carbon source (as comma-separated compound names)
- Nitrogen source (as comma-separated compound names)

In [ ]:
def flatten_carbon_source(media_spec):
    """Extract carbon source as comma-separated string."""
    if media_spec == 'unspecified':
        return 'unspecified'
    
    carbon = media_spec.get('carbon_source', 'unspecified')
    if carbon == 'unspecified':
        return 'unspecified'
    
    if isinstance(carbon, list):
        compounds = [c['compound'] for c in carbon]
        return ', '.join(compounds)
    
    return str(carbon)

def flatten_nitrogen_source(media_spec):
    """Extract nitrogen source as comma-separated string."""
    if media_spec == 'unspecified':
        return 'unspecified'
    
    nitrogen = media_spec.get('nitrogen_source', 'unspecified')
    if nitrogen == 'unspecified':
        return 'unspecified'
    
    if isinstance(nitrogen, list):
        compounds = [n['compound'] for n in nitrogen]
        return ', '.join(compounds)
    
    return str(nitrogen)

# Create flattened mapping for all conditions
flattened_media = {}
for condition, media_spec in media_specs.items():
    flattened_media[condition] = {
        'media_name': media_spec.get('name', 'unspecified') if media_spec != 'unspecified' else 'unspecified',
        'carbon_source': flatten_carbon_source(media_spec),
        'nitrogen_source': flatten_nitrogen_source(media_spec)
    }

# Display flattened mapping
print("Flattened media specifications:\n")
for condition in sorted(flattened_media.keys()):
    print(f"{condition}:")
    for key, value in flattened_media[condition].items():
        print(f"  {key}: {value}")
    print()

## 5. Build SQL CASE Expressions Manually

Now we'll construct SQL CASE expressions to map each condition value to its media specifications. This is the core of creating enriched metadata.

In [ ]:
# Build CASE expression for media_name
media_name_cases = []
for condition, specs in sorted(flattened_media.items()):
    media_name_cases.append(f"        WHEN '{condition}' THEN '{specs['media_name']}'")
media_name_sql = "\n".join(media_name_cases)

# Build CASE expression for carbon_source
carbon_source_cases = []
for condition, specs in sorted(flattened_media.items()):
    carbon_source_cases.append(f"        WHEN '{condition}' THEN '{specs['carbon_source']}'")
carbon_source_sql = "\n".join(carbon_source_cases)

# Build CASE expression for nitrogen_source
nitrogen_source_cases = []
for condition, specs in sorted(flattened_media.items()):
    nitrogen_source_cases.append(f"        WHEN '{condition}' THEN '{specs['nitrogen_source']}'")
nitrogen_source_sql = "\n".join(nitrogen_source_cases)

# Construct the complete CREATE VIEW SQL
create_view_sql = f"""
CREATE OR REPLACE VIEW harbison_2004_enriched_metadata AS
SELECT
    sample_id,
    regulator_locus_tag,
    regulator_symbol,
    condition,
    CASE condition
{media_name_sql}
    END AS media_name,
    CASE condition
{carbon_source_sql}
    END AS carbon_source,
    CASE condition
{nitrogen_source_sql}
    END AS nitrogen_source
FROM harbison_2004_train
"""

print("Generated SQL:")
print("="*80)
print(create_view_sql)
print("="*80)

## 6. Execute SQL and Create Enriched Metadata View

Now let's execute the SQL to create our enriched metadata view.

In [ ]:
# Execute the CREATE VIEW statement
conn.execute(create_view_sql)

print("Enriched metadata view created successfully!")
print("\nAvailable views:")
views = conn.execute("SHOW TABLES").fetchall()
for view in views:
    print(f"  - {view[0]}")

### Query the Enriched Metadata View

Let's examine the enriched metadata to see the results.

In [ ]:
# Query enriched metadata
enriched_df = conn.execute("""
    SELECT *
    FROM harbison_2004_enriched_metadata
    LIMIT 10
""").df()

print(f"Enriched metadata shape: {enriched_df.shape}")
print("\nSample rows:")
display(enriched_df)

## 7. Practical Examples: Querying with Enriched Metadata

Now that we have enriched metadata, we can perform analyses that wouldn't be possible with just the raw condition values.

### Example 1: Find All Experiments with D-glucose as Carbon Source

In [ ]:
glucose_df = conn.execute("""
    SELECT DISTINCT
        condition,
        media_name,
        carbon_source,
        nitrogen_source,
        COUNT(*) as sample_count
    FROM harbison_2004_enriched_metadata
    WHERE carbon_source LIKE '%D-glucose%'
    GROUP BY condition, media_name, carbon_source, nitrogen_source
    ORDER BY condition
""").df()

print(f"Found {len(glucose_df)} condition(s) with D-glucose")
display(glucose_df)

### Example 2: Compare Rich Media vs Synthetic Media Experiments

In [ ]:
media_comparison_df = conn.execute("""
    SELECT
        CASE
            WHEN media_name LIKE '%YPD%' OR media_name LIKE '%yeast_extract%'
            THEN 'Rich Media'
            WHEN media_name LIKE '%synthetic%'
            THEN 'Synthetic Media'
            ELSE 'Other'
        END AS media_type,
        COUNT(DISTINCT condition) as num_conditions,
        COUNT(DISTINCT sample_id) as num_samples,
        COUNT(DISTINCT regulator_symbol) as num_regulators
    FROM harbison_2004_enriched_metadata
    GROUP BY media_type
    ORDER BY media_type
""").df()

print("Media type comparison:")
display(media_comparison_df)

### Example 3: Analyze Carbon Source Diversity

In [ ]:
carbon_diversity_df = conn.execute("""
    SELECT
        carbon_source,
        COUNT(DISTINCT condition) as num_conditions,
        COUNT(DISTINCT sample_id) as num_samples,
        STRING_AGG(DISTINCT condition, ', ' ORDER BY condition) as conditions
    FROM harbison_2004_enriched_metadata
    GROUP BY carbon_source
    ORDER BY num_conditions DESC
""").df()

print("Carbon source diversity:")
display(carbon_diversity_df)

### Example 4: Find Regulators Tested Across Multiple Carbon Sources

In [ ]:
regulator_diversity_df = conn.execute("""
    SELECT
        regulator_symbol,
        regulator_locus_tag,
        COUNT(DISTINCT carbon_source) as num_carbon_sources,
        COUNT(DISTINCT condition) as num_conditions,
        STRING_AGG(DISTINCT carbon_source, ', ') as carbon_sources_tested
    FROM harbison_2004_enriched_metadata
    GROUP BY regulator_symbol, regulator_locus_tag
    HAVING COUNT(DISTINCT carbon_source) > 1
    ORDER BY num_carbon_sources DESC
    LIMIT 20
""").df()

print(f"Found {len(regulator_diversity_df)} regulators tested across multiple carbon sources")
print("\nTop 20 regulators by carbon source diversity:")
display(regulator_diversity_df)

## 8. Summary and Key Takeaways

In this tutorial, we learned how to:

1. **Explore nested metadata** using DataCard's `get_field_attribute()` method
2. **Flatten nested structures** into queryable strings
3. **Build SQL CASE expressions** manually to map condition values to metadata attributes
4. **Create enriched metadata views** that combine data fields with DataCard information
5. **Query by metadata attributes** that aren't present in the raw data

### When to Use This Approach

Use this pattern when:
- You need to analyze data by attributes that are defined in the DataCard but not in the data itself
- You want full control over the SQL being generated
- You're working with a single dataset and need custom metadata columns

### Next Steps

- Explore other attributes like `temperature_celsius` or `cultivation_method`
- Combine multiple attributes in your enriched views
- Use this pattern with other datasets in the BrentLab collection
- Build reusable helper functions for your specific analysis needs